---
title: "Earnings Call Transcript Analysis"
subtitle: "Assignment 1 · Documentation · Validation · Exploratory Data Analysis"
date: today
format:
  html:
    theme: darkly
    toc: true
    toc-location: left
    toc-depth: 3
    toc-title: "Contents"
    code-fold: true
    code-summary: "▸ show code"
    embed-resources: true
    self-contained: true
    smooth-scroll: true
    fig-width: 9
    fig-height: 5
    include-in-header:
      text: |
        <link rel="preconnect" href="https://fonts.googleapis.com">
        <link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap" rel="stylesheet">
execute:
  echo: true
  warning: false
  message: false
  cache: true
editor_options: 
  chunk_output_type: console
---

```{r venv-create}
#| eval: false
#| cache: false

# Run this chunk once manually (Cmd+Shift+Enter or click Run)
# Never runs during quarto render — eval: false

reticulate::virtualenv_create("earnings")

reticulate::virtualenv_install("earnings", packages = c(
  "pandas",
  "numpy",
  "transformers",
  "torch",
  "gensim",
  "sentence-transformers",
  "umap-learn"
))

# Verify — should print the venv's python path
reticulate::virtualenv_python("earnings")
```

```{r venv-activate}
#| cache: false

reticulate::use_virtualenv("earnings", required = TRUE)

# Sanity check — prints during render so you can confirm the right Python loaded
reticulate::py_config()

```



```{=html}
<style>
/* ============================================================
   Dark-Inspired Material Dark Theme — Quarto Port
   ============================================================ */

:root {
  --bg-primary:    #0d0d14;
  --bg-secondary:  #13131e;
  --bg-card:       #1a1a28;
  --bg-hover:      #22223a;
  --accent-gold:   #f5a623;
  --accent-red:    #eb001b;
  --accent-orange: #ff6b35;
  --text-primary:  #e8e8f0;
  --text-muted:    #8888aa;
  --border:        #2a2a42;
}

body, html {
  background-color: var(--bg-primary) !important;
  font-family: 'DM Sans', sans-serif !important;
  color: var(--text-primary) !important;
}

/* ── Navbar ── */
.navbar {
  background: var(--bg-secondary) !important;
  border-bottom: 1px solid var(--border) !important;
}
.navbar-brand, .navbar-brand:hover {
  color: var(--text-primary) !important;
  font-size: 12px !important;
  font-weight: 600 !important;
  letter-spacing: 0.08em !important;
  text-transform: uppercase !important;
}

/* ── Left TOC ── */
#quarto-sidebar, .sidebar {
  background: var(--bg-secondary) !important;
  border-right: 1px solid var(--border) !important;
}
.sidebar-title { color: var(--accent-gold) !important; font-size: 11px !important; font-weight: 600 !important; letter-spacing: 0.1em !important; text-transform: uppercase !important; }
.sidebar-item-text { color: var(--text-primary) !important; font-size: 12px !important; }
.sidebar-item-text:hover, .active .sidebar-item-text { color: #ffffff !important; }
.sidebar-item.active > .sidebar-item-container > .sidebar-item-text { color: var(--accent-gold) !important; }

/* ── Page body ── */
#quarto-content { background: var(--bg-primary) !important; }
.quarto-title { border-bottom: 1px solid var(--border); padding-bottom: 1rem; margin-bottom: 2rem; }
h1, h2, h3, h4 { color: var(--text-primary) !important; }
h2 { border-bottom: 1px solid var(--border); padding-bottom: .4rem; margin-top: 2.5rem; font-size: 1.1rem; letter-spacing: .06em; text-transform: uppercase; }
h3 { color: var(--accent-gold) !important; font-size: 0.95rem; margin-top: 2rem; }
p, li { color: var(--text-primary) !important; font-size: 13px; line-height: 1.75; }
strong { color: #ffffff !important; }

/* ── Code blocks ── */
pre, code { font-family: 'DM Mono', monospace !important; font-size: 12px !important; }
pre { background: var(--bg-card) !important; border: 1px solid var(--border) !important; border-radius: 6px !important; }
.code-fold-btn { color: var(--accent-gold) !important; font-size: 11px !important; border: 1px solid var(--border) !important; background: var(--bg-card) !important; border-radius: 4px !important; padding: 2px 8px !important; }

/* ── Callouts ── */
.callout { background: var(--bg-card) !important; border-left-color: var(--accent-gold) !important; border: 1px solid var(--border) !important; border-radius: 6px !important; }
.callout-title { color: var(--accent-gold) !important; font-size: 11px !important; font-weight: 600 !important; text-transform: uppercase !important; letter-spacing: .08em !important; }
.callout-body p { color: var(--text-primary) !important; }

/* ── Tables ── */
table { border-collapse: collapse !important; width: 100% !important; font-size: 12px !important; }
thead tr { background: var(--bg-secondary) !important; border-bottom: 1px solid var(--border) !important; }
thead th { color: var(--accent-gold) !important; font-weight: 600 !important; letter-spacing: .06em !important; text-transform: uppercase !important; font-size: 10px !important; padding: 8px 12px !important; }
tbody tr { border-bottom: 1px solid var(--border) !important; }
tbody tr:hover { background: var(--bg-hover) !important; }
tbody td { color: var(--text-primary) !important; padding: 7px 12px !important; }

/* ── Python output terminal card ── */
.cell-output pre {
  background: var(--bg-card) !important;
  border: 1px solid var(--border) !important;
  border-left: 3px solid var(--accent-gold) !important;
  border-radius: 6px !important;
  padding: 14px 16px !important;
  color: var(--text-primary) !important;
  font-size: 12px !important;
  line-height: 1.6 !important;
}

/* ── Highcharts container ── */
.highcharts-container, .highcharts-root { border-radius: 6px !important; }

/* ── wordcloud2 container ── */
#wordcloud-wrap { background: var(--bg-primary); border-radius: 6px; border: 1px solid var(--border); padding: 8px; }

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: var(--bg-primary); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent-gold); }
</style>
```




```{r setup}
#| label: setup
#| cache: false
#| cache.lazy: false
#| echo: true

# ── Libraries ─────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
  library(tidyverse)
  library(tidytext)
  library(topicmodels)
  library(highcharter)
  library(htmltools)
  library(tm)
  library(slam)
  library(scales)
  library(glue)
  library(zoo)
  library(lubridate)
  library(reticulate)
  library(knitr)
  library(kableExtra)
})

here_path <- here::here('analytical_ml_finance')

# ── Colour palettes ────────────────────────────────────────────────────────────
pal8 <- c("#f5a623", "#ff6b35", "#4ecdc4", "#a78bfa",
          "#34d399", "#f472b6", "#60a5fa", "#fb923c")
pal9 <- c(pal8, "#e879f9")

cat_palette <- c(
  positive     = "#34d399",
  negative     = "#f87171",
  uncertainty  = "#f5a623",
  litigious    = "#a78bfa",
  constraining = "#60a5fa"
)

# ── Dark dark highcharter theme ─────────────────────────────────────────────────
my_theme <- hc_theme(
  colors = pal8,
  chart  = list(
    backgroundColor = "#0d0d14",
    style = list(fontFamily = "DM Sans, sans-serif", color = "#e8e8f0")
  ),
  title    = list(style = list(color = "#e8e8f0", fontSize = "14px", fontWeight = "600")),
  subtitle = list(style = list(color = "#c8c8d8", fontSize = "11px")),
  xAxis = list(
    gridLineColor = "#1e1e30", lineColor = "#2a2a42", tickColor = "#2a2a42",
    labels = list(style = list(color = "#e8e8f0", fontSize = "11px")),
    title  = list(style = list(color = "#e8e8f0", fontSize = "11px"))
  ),
  yAxis = list(
    gridLineColor = "#1e1e30", lineColor = "transparent", tickColor = "transparent",
    labels = list(style = list(color = "#e8e8f0", fontSize = "11px")),
    title  = list(style = list(color = "#e8e8f0", fontSize = "11px"))
  ),
  legend = list(
    itemStyle      = list(color = "#e8e8f0", fontSize = "11px", fontWeight = "400"),
    itemHoverStyle = list(color = "#ffffff")
  ),
  tooltip = list(
    backgroundColor = "#1a1a28", borderColor = "#2a2a42", borderRadius = 6,
    style = list(color = "#e8e8f0", fontSize = "12px")
  ),
  plotOptions = list(
    series = list(
      dataLabels = list(style = list(color = "#e8e8f0", textOutline = "none"))
    )
  ),
  credits = list(enabled = FALSE)
)

# ── Top-50 S&P 500 tickers (by index weight) ──────────────────────────────────
tickers_top50 <- tidyquant::tq_index("SP500") |>
  arrange(desc(weight)) |>
  slice(1:50) |>
  pull(symbol)

# ── Data load ─────────────────────────────────────────────────────────────────
earnings_calls_raw <- vroom::vroom(
  path.expand("~/Downloads/Raw Transcripts/aggregated_all_years.csv")
) |>
  mutate(
    quarter        = as.Date(zoo::as.yearqtr(mostimportantdateutc)),
    fiscal_year    = year(quarter),
    fiscal_quarter = paste0("Q", quarter(quarter))
  ) 

earnings_calls <- earnings_calls_raw |>
  filter(ticker %in% tickers_top50)


# ── Text cleaning ──────────────────────────────────────────────────────────────
clean_text <- function(txt) {
  txt |>
    str_remove_all("\\[Operator Instructions\\]") |>
    str_remove_all("(?i)forward-looking statements.*?filings\\.") |>
    str_squish()
}

# ── Tokenisation ───────────────────────────────────────────────────────────────
# Full corpus — lightweight ops (wordcloud, global bag-of-words)


# Top-50 filtered — all downstream NLP
tokens <- earnings_calls |>
  mutate(doc_id = row_number(), text = clean_text(full_transcript_text)) |>
  select(doc_id, ticker, companyname, fiscal_year, fiscal_quarter, text) |>
  unnest_tokens(word, text) |>
  anti_join(stop_words, by = "word") |>
  filter(str_detect(word, "^[a-z]+$"), nchar(word) > 2)

# ── Forward-looking sentence detection ────────────────────────────────────────
fwd_patterns <- paste0(
  "(?i)\\b(expect|anticipate|project|forecast|outlook|guidance|",
  "will|plan to|intend|believe|estimate|target|aim|hope|",
  "going forward|in the coming|remainder of|full.?year|next quarter|",
  "by year.?end|longer.?term|over the next)\\b"
)

sent_sentences <- earnings_calls |>
  mutate(doc_id = row_number()) |>
  unnest_tokens(sentence, full_transcript_text, token = "sentences") |>
  mutate(is_forward = str_detect(sentence, fwd_patterns))

# ── Loughran-McDonald sentiment ────────────────────────────────────────────────
lm_sentiment <- get_sentiments("loughran")

sentiment_doc <- tokens |>
  inner_join(lm_sentiment, by = "word") |>
  count(doc_id, sentiment) |>
  pivot_wider(names_from = sentiment, values_from = n, values_fill = 0) |>
  mutate(net = (positive - negative) / pmax(positive + negative, 1)) |>
  left_join(
    earnings_calls |>
      mutate(doc_id = row_number()) |>
      select(doc_id, ticker, fiscal_year, fiscal_quarter),
    by = "doc_id"
  )

# ── Pre-compute doc-level stats for inline use ─────────────────────────────────
n_raw            <- nrow(earnings_calls_raw)
n_with_text      <- sum(!is.na(earnings_calls_raw$full_transcript_text) &
                          nchar(earnings_calls_raw$full_transcript_text) > 100)
n_with_permno    <- sum(!is.na(earnings_calls_raw$permno))
n_unique_firms   <- n_distinct(earnings_calls_raw$ticker)
n_unique_permnos <- n_distinct(na.omit(earnings_calls_raw$permno))
date_min         <- min(earnings_calls_raw$quarter, na.rm = TRUE)
date_max         <- max(earnings_calls_raw$quarter, na.rm = TRUE)
permno_rate      <- round(100 * n_with_permno / n_raw, 1)
```

---

## 1 · Data Documentation

### 1.1 Dataset Overview

This dataset compiles earnings conference call transcripts for publicly traded U.S. equities. The transcripts were sourced from a structured text data provider, and supplemented with market-microstructure fields (same-day close, next-day open) to enable downstream return-prediction research. CRSP PERMNO identifiers are included to facilitate merges with standard financial databases.

```{r doc-table}
#| echo: false

tibble(
  Field   = c("Source file", "Time range", "Total transcripts",
               "Unique tickers", "Unique PERMNOs", "PERMNO match rate",
               "NLP scope (this analysis)", "Key text field"),
  Value   = c(
    "aggregated_all_years.csv",
    paste(format(date_min, "%b %Y"), "–", format(date_max, "%b %Y")),
    format(n_raw, big.mark = ","),
    format(n_unique_firms, big.mark = ","),
    format(n_unique_permnos, big.mark = ","),
    paste0(permno_rate, "%"),
    paste0("Top-50 S&P 500 constituents by index weight (n = ",
           format(nrow(earnings_calls), big.mark = ","), ")"),
    "full_transcript_text"
  )
) |>
  kable(col.names = c("Field", "Value"), align = c("l","l")) |>
  kable_styling(bootstrap_options = c("striped","hover","condensed"),
                full_width = FALSE, position = "left", font_size = 12)
```

### 1.2 Column Reference

```{r col-ref}
#| echo: false

tibble(
  Column = c("transcriptid","companyid","companyname","ticker","permno",
             "mostimportantdateutc","call_date","actual_call_date",
             "quarter","fiscal_year","fiscal_quarter",
             "full_transcript_text","word_count","transcript_length",
             "close_price_call_day","open_price_next_day","close_to_open_return",
             "event_type"),
  Type   = c("int","int","chr","chr","dbl",
             "chr","chr","chr",
             "date","int","chr",
             "chr","int","int",
             "dbl","dbl","dbl",
             "chr"),
  Description = c(
    "Provider-assigned transcript identifier",
    "Provider-assigned company identifier",
    "Full legal company name",
    "Exchange ticker symbol",
    "CRSP permanent number — enables CRSP/Compustat merge",
    "UTC timestamp of transcript creation",
    "Scheduled call date",
    "Confirmed call date (may differ from scheduled)",
    "First day of the fiscal quarter (derived via zoo::as.yearqtr)",
    "Calendar year of the call (derived)",
    "Fiscal quarter label Q1–Q4 (derived)",
    "Full verbatim transcript text",
    "Token count of transcript",
    "Character length of transcript",
    "CRSP closing price on the call date",
    "CRSP opening price the following trading day",
    "Log return: open_next / close_call − 1",
    "Call classification (Earnings Calls, etc.)"
  )
) |>
  kable(col.names = c("Column","Type","Description"), align = c("l","l","l")) |>
  kable_styling(bootstrap_options = c("striped","hover","condensed"),
                full_width = TRUE, font_size = 11)
```

### 1.3 Derived Fields

The `quarter`, `fiscal_year`, and `fiscal_quarter` columns are not present in the raw file and are constructed as follows:

```r
mutate(
  quarter        = as.Date(zoo::as.yearqtr(mostimportantdateutc)),
  fiscal_year    = lubridate::year(quarter),
  fiscal_quarter = paste0("Q", lubridate::quarter(quarter))
)
```

---

## 2 · Validation Report

### 2.1 Completeness & Matching

```{r validation-r}
#| echo: true

# ── Completeness metrics ───────────────────────────────────────────────────────
valid_summary <- tibble(
  Metric = c(
    "Raw transcripts",
    "Transcripts with text (> 100 chars)",
    "Transcripts with valid PERMNO",
    "Transcripts with price data (close_price_call_day)",
    "Transcripts with return data (close_to_open_return)",
    "Records dropped — missing text",
    "Records dropped — missing PERMNO"
  ),
  N = c(
    n_raw,
    n_with_text,
    n_with_permno,
    sum(!is.na(earnings_calls_raw$close_price_call_day)),
    sum(!is.na(earnings_calls_raw$close_to_open_return)),
    n_raw - n_with_text,
    n_raw - n_with_permno
  )
) |>
  mutate(
    `Share of Raw` = ifelse(
      str_detect(Metric, "dropped"),
      paste0(round(100 * N / n_raw, 1), "%  ↓"),
      paste0(round(100 * N / n_raw, 1), "%")
    ),
    N = format(N, big.mark = ",")
  )

kable(valid_summary, align = c("l","r","r")) |>
  kable_styling(bootstrap_options = c("striped","hover","condensed"),
                full_width = FALSE, position = "left", font_size = 12)
```

### 2.2 Transcript Length Distribution

```{r validation-length}
# Scope: full corpus — computationally negligible
length_quantiles <- earnings_calls_raw |>
  filter(!is.na(word_count)) |>
  summarise(
    p5  = quantile(word_count, .05),
    p25 = quantile(word_count, .25),
    p50 = quantile(word_count, .50),
    p75 = quantile(word_count, .75),
    p95 = quantile(word_count, .95),
    mean = mean(word_count),
    sd   = sd(word_count)
  ) |>
  pivot_longer(everything(), names_to = "Statistic", values_to = "Word Count") |>
  mutate(`Word Count` = round(`Word Count`))

kable(length_quantiles, align = c("l","r")) |>
  kable_styling(bootstrap_options = c("striped","hover","condensed"),
                full_width = FALSE, position = "left", font_size = 12)
```

### 2.3 Cleaning Steps & Limitations

::: {.callout-note}
**Cleaning decisions applied to all downstream analysis:**

1. **Boilerplate removal** — Operator instruction tags `[Operator Instructions]` and legal safe-harbour preambles matching `forward-looking statements.*filings\.` are stripped before tokenisation.
2. **Stopword removal** — Standard tidytext English stopwords are removed. Financial stop-words (e.g., *company*, *quarter*) are handled implicitly by TF-IDF down-weighting.
3. **Token filter** — Only lowercase alphabetic tokens of length > 2 are retained. Numeric strings, ticker fragments, and punctuation artifacts are excluded.
4. **Length filter** — Transcripts with fewer than 100 characters in `full_transcript_text` are treated as unusable and excluded from NLP.
5. **PERMNO gaps** — Approximately `r paste0(100 - permno_rate, "%")` of transcripts lack a matched PERMNO. These are retained in EDA but must be excluded from any return-regression analysis.
6. **Scope trim** — All heavy NLP (LDA, sentiment, TF-IDF) is restricted to the top-50 S&P 500 constituents by current index weight to ensure runtime feasibility. Bag-of-words and word frequency operations use the full corpus.
:::


---

## 3 · Exploratory Data Analysis

::: {.callout-note}
**NLP scope note:** Sections 3.3 – 3.9 are restricted to the **top-50 S&P 500 firms by index weight** (`n = {r format(nrow(earnings_calls_raw), big.mark=",")} transcripts`). Section 3.2 (Word Cloud) uses the full corpus.
:::

### 3.1 Corpus Coverage

```{r coverage}
by_year <- earnings_calls_raw |> count(fiscal_year, name = "n")

by_firm <- earnings_calls_raw |> unite(ticker,companyid,ticker) |>
  count(ticker, sort = TRUE) |>
  slice_head(n = 30) |>
  mutate(ticker = fct_reorder(ticker, n))

hc_year <- hchart(by_year, "column", hcaes(x = fiscal_year, y = n),
                  name = "Transcripts",
                  color = "#f5a623",
                  borderRadius = 3,
                  dataLabels = list(
                    enabled = TRUE, format = "{y}",
                    style = list(fontSize = "9px", color = "#e8e8f0",
                                 textOutline = "none")
                  )) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Transcripts by Year — Full Corpus") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title = list(text = "# Transcripts")) |>
  hc_tooltip(pointFormat = "<b>{point.x}</b>: {point.y} transcripts")

hc_firm <- hchart(by_firm, "bar", hcaes(x = ticker, y = n),
                  name = "Transcripts",
                  color = "#4ecdc4",
                  borderRadius = 2,
                  dataLabels = list(
                    enabled = TRUE, format = "{y}",
                    style = list(fontSize = "8px", color = "#e8e8f0",
                                 textOutline = "none")
                  )) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Top 30 Firms by Coverage") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title = list(text = "# Transcripts")) |>
  hc_chart(height = 550)


```


:::{layout-ncol=2}
```{r}
hc_year
```
```{r}
hc_firm
```
:::


> **Interpretation.** The by-year chart reveals whether the panel has collection gaps or ramp-up artefacts — uneven coverage should be controlled via year fixed effects in any regression. The by-firm chart shows coverage concentration: heavy overlap with mega-cap names is expected and worth flagging as a generalisability limitation.

### 3.2 Corpus Word Cloud

Filtered term frequency after stopword removal. Size encodes raw count. Computed over the filtered data set (top 50 firms).

```{r wordcloud}
#| fig-height: 4.5

word_freq <- tokens |>
  count(word, sort = TRUE) |>
  slice_head(n = 250) |>
  rename(freq = n)

hchart(word_freq, "wordcloud", hcaes(name = word, weight = freq)) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Corpus Word Cloud") |>
  hc_subtitle(text = "Top 200 terms after stopword removal — full corpus") |>
  hc_tooltip(pointFormat = "<b>{point.name}</b>: {point.weight:,.0f}") %>% 
  hc_add_dependency("modules/wordcloud.js")
```

### 3.3 Keyword Trends

```{r keywords}
keywords <- c("revenue", "guidance", "margin", "inflation", "recession",
              "dividend", "interest", "uncertainty", "headwind", "tailwind",
              "growth", "outlook", "capex", "debt")

kw_time <- tokens |>
  filter(word %in% keywords) |>
  count(fiscal_year, word) |>
  group_by(fiscal_year) |>
  mutate(share = n / sum(n) * 100) |>
  ungroup()

hchart(kw_time, "spline", hcaes(x = fiscal_year, y = share, group = word)) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Key Financial Concept Share of Tokens Over Time") |>
  hc_subtitle(text = "% of keyword tokens (top-50 corpus) — drag to zoom, click legend to isolate") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title  = list(text = "% of keyword tokens"),
           labels = list(format = "{value}%")) |>
  hc_tooltip(shared = TRUE, crosshairs = TRUE,
             valueSuffix = "%", valueDecimals = 2) |>
  hc_legend(align = "right", layout = "proximate") |>
  hc_chart(zoomType = "x") |>
  hc_plotOptions(spline = list(
    lineWidth = 2,
    marker    = list(radius = 3, symbol = "circle"),
    states    = list(hover = list(lineWidth = 3))
  ))
```



> **Interpretation.** Spikes in *inflation* and *recession* around 2022–2023 reflect macro stress. Rising *uncertainty* and *headwind* language typically co-moves with guidance cuts. *Guidance* frequency is a useful leading indicator of forward-looking disclosure quantity.

### 3.4 TF–IDF by Firm

Top-8 TF-IDF terms per firm for the 9 highest-coverage firms. TF-IDF surfaces terms that are frequent within a firm but rare corpus-wide — a signature vocabulary proxy.

```{r tfidf}
focus_tickers <- earnings_calls |>
  count(ticker, sort = TRUE) |>
  slice_head(n = 9) |>
  pull(ticker)

tfidf_firm <- tokens |>
  count(ticker, word, sort = TRUE) |>
  bind_tf_idf(word, ticker, n) |>
  filter(ticker %in% focus_tickers) |>
  group_by(ticker) |>
  slice_max(tf_idf, n = 8) |>
  ungroup()

charts_tfidf <- map2(focus_tickers, pal9[seq_along(focus_tickers)], function(tick, col) {
  d <- tfidf_firm |>
    filter(ticker == tick) |>
    arrange(tf_idf) |>
    mutate(word = factor(word, levels = word))

  hchart(d, "bar", hcaes(x = word, y = tf_idf),
         name = "TF-IDF", color = col, borderRadius = 2,
         dataLabels = list(
           enabled = TRUE, format = "{point.y:.3f}",
           style = list(fontSize = "8px", color = "#e8e8f0", textOutline = "none")
         )) |>
    hc_add_theme(my_theme) |>
    hc_title(text = tick, style = list(fontSize = "12px", color = col)) |>
    hc_xAxis(title = list(text = "")) |>
    hc_yAxis(title = list(text = ""), visible = FALSE) |>
    hc_size(height = 230) |>
    hc_tooltip(pointFormat = "<b>{point.x}</b>: {point.y:.4f}")
})

# tagList(tags$div(
#   style = "display:grid; grid-template-columns:repeat(3, 1fr); gap:10px;",
#   charts_tfidf
# ))
```


:::{layout-ncol=3}
```{r}
charts_tfidf[[1]]
```
```{r}
charts_tfidf[[2]]
```
```{r}
charts_tfidf[[3]]
```
```{r}
charts_tfidf[[4]]
```
```{r}
charts_tfidf[[5]]
```
```{r}
charts_tfidf[[6]]
```
```{r}
charts_tfidf[[7]]
```
```{r}
charts_tfidf[[8]]
```
```{r}
charts_tfidf[[9]]
```
:::

> **Interpretation.** Tech-heavy names surface product vocabulary; regulated financials surface legal/compliance terms. This view is a useful sanity check before topic modelling — if TF-IDF terms are economically coherent, LDA topics are likely to be as well.

### 3.5 LDA Topic Discovery

```{r lda-setup}
#| cache: true

set.seed(42)
n_topics <- 12

if(file.exists('lda_model.rds')){

  lda_model_obj <- readRDS("lda_model.rds")

  lda_model <- lda_model_obj$lda_model
  lda_terms <- lda_model_obj$lda_terms
  lda_docs <- lda_model_obj$lda_docs
  
}else{

  dtm <- tokens |>
  add_count(word, name = "corpus_n") |>
  filter(
    corpus_n >= 10,
    corpus_n <= 0.85 * n_distinct(doc_id) * 5
  ) |>
  count(doc_id, word) |>
  cast_dtm(doc_id, word, n)

dtm <- dtm[slam::row_sums(dtm) > 0, ]
dtm  <- removeSparseTerms(dtm, 0.998)

lda_model <- LDA(dtm, k = n_topics,
                 method  = "VEM",
                 control = list(seed = 1:5, nstart = 5))

lda_terms <- tidy(lda_model, matrix = "beta") |>
  group_by(topic) |>
  slice_max(beta, n = 10) |>
  ungroup()

lda_docs <- tidy(lda_model, matrix = "gamma") |>
  mutate(document = as.integer(document)) |>
  left_join(
    earnings_calls_raw |>
      mutate(document = row_number()) |>
      select(document, ticker, fiscal_year),
    by = "document"
  )

lda_model_obj <- list("lda_model" = lda_model,"lda_docs" = lda_docs,"lda_terms" = lda_terms)

saveRDS(lda_model_obj,"lda_model.rds")




    
}

topic_labels <- c(
  "1" = "Software and Computing",   "2" = "Consumer Payments",
  "3" = "Media and Broadband",      "4" = "Healthcare",
  "5" = "Banking",  "6" = "Random",
  "7" = "Production",   "8" = "Brand", "9" = "Healthcare / Pharma", "10" = "Network security", "11" = "Operations","12" = "Consumer Tech"
)


```

```{r lda-chart}
charts_lda <- map(1:n_topics, function(k) {
  d <- lda_terms |>
    filter(topic == k) |>
    arrange(beta) |>
    mutate(term = factor(term, levels = term))

  hchart(d, "bar", hcaes(x = term, y = beta),
         name = "β", color = pal8[k], borderRadius = 2,
         dataLabels = list(
           enabled = TRUE, format = "{point.y:.4f}",
           style = list(fontSize = "7px", color = "#e8e8f0", textOutline = "none")
         )) |>
    hc_add_theme(my_theme) |>
    hc_title(text  = glue("T{k}: {topic_labels[as.character(k)]}"),
             style = list(fontSize = "11px", color = pal8[k])) |>
    hc_xAxis(title = list(text = "")) |>
    hc_yAxis(title = list(text = ""), visible = FALSE) |>
    hc_tooltip(pointFormat = "<b>{point.x}</b>: β = {point.y:.5f}")
})

# tagList(tags$div(
#   style = "display:grid; grid-template-columns:repeat(4, 1fr); gap:10px;",
#   charts_lda
# ))
```


:::{layout-ncol=3}
```{r}
charts_lda[[1]]
```
```{r}
charts_lda[[2]]
```
```{r}
charts_lda[[3]]
```
```{r}
charts_lda[[4]]
```
```{r}
charts_lda[[5]]
```
```{r}
charts_lda[[6]]
```
```{r}
charts_lda[[7]]
```
```{r}
charts_lda[[8]]
```
```{r}
charts_lda[[9]]
```
```{r}
charts_lda[[10]]
```
```{r}
charts_lda[[11]]
```
```{r}
charts_lda[[12]]
```
:::


> **Interpretation.** LDA (k=8, VEM) decomposes the corpus into co-occurring word clusters. Each panel shows the top β-weighted terms — the probability a word is drawn from that topic. Topic labels are interpretive and should be revisited after examining full top-20 term lists. Tune k by plotting held-out perplexity over k ∈ {4, …, 14} before finalising.

### 3.6 Topic Mix Over Time

```{r topic-time}
topic_time <- lda_docs |>
  group_by(document, fiscal_year) |>
  slice_max(gamma, n = 1, with_ties = FALSE) |>
  ungroup() |>
  count(fiscal_year, topic) |>
  group_by(fiscal_year) |>
  mutate(pct = n / sum(n) * 100) |>
  ungroup() |>
  mutate(label = factor(topic_labels[as.character(topic)],
                        levels = unname(topic_labels)))

hchart(topic_time, "area", hcaes(x = fiscal_year, y = pct, group = label)) |>
  hc_add_theme(my_theme) |>
  hc_plotOptions(area = list(
    stacking    = "normal",
    lineWidth   = 0.5,
    fillOpacity = 0.78,
    marker      = list(enabled = FALSE)
  )) |>
  hc_title(text = "Topic Prevalence Over Time") |>
  hc_subtitle(text = "Share of dominant topic per transcript (argmax γ) — top-50 corpus") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title  = list(text = "% of transcripts"),
           labels = list(format = "{value}%"),
           max    = 100) |>
  hc_tooltip(shared = TRUE, crosshairs = TRUE,
             valueSuffix = "%", valueDecimals = 1) |>
  hc_legend(layout = "proximate",align = "right")
```

> **Interpretation.** A 100%-stacked area chart reveals how discourse mix shifts year-by-year. Increasing macro-environment share around 2022 and rising digital/services share post-2020 would validate the LDA solution as economically meaningful. Extend by computing probability-weighted topic shares (rather than argmax) for a smoother signal.

### 3.7 Forward-Looking Language

*Note: the IQR band + mean-line overlay requires `hc_add_series` — this is the one chart where the arearange type cannot be expressed as a single `hchart()` call.*

```{r forward}
fwd_year <- sent_sentences |>
  group_by(doc_id, fiscal_year) |>
  summarise(fwd_share = mean(is_forward) * 100, .groups = "drop") |>
  group_by(fiscal_year) |>
  summarise(
    mean_fwd = mean(fwd_share),
    q25      = quantile(fwd_share, 0.25),
    q75      = quantile(fwd_share, 0.75),
    .groups  = "drop"
  )

highchart() |>
  hc_add_theme(my_theme) |>
  hc_chart(zoomType = "x") |>
  hc_title(text = "Forward-Looking Language Over Time") |>
  hc_subtitle(text = "% of sentences containing prospective language — mean ± IQR (top-50 corpus)") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title  = list(text = "% sentences forward-looking"),
           labels = list(format = "{value}%")) |>
  hc_tooltip(shared = TRUE, crosshairs = TRUE,
             valueSuffix = "%", valueDecimals = 1) |>
  hc_add_series(
    name        = "IQR Band",
    type        = "arearange",
    color       = "#f5a623",
    fillOpacity = 0.15,
    lineWidth   = 0,
    marker      = list(enabled = FALSE),
    data = map(seq_len(nrow(fwd_year)), ~ list(
      fwd_year$fiscal_year[.x],
      round(fwd_year$q25[.x], 2),
      round(fwd_year$q75[.x], 2)
    ))
  ) |>
  hc_add_series(
    name      = "Mean",
    type      = "spline",
    color     = "#f5a623",
    lineWidth = 2.5,
    marker    = list(radius = 4, symbol = "circle",
                     fillColor = "#f5a623", lineColor = "#0d0d14", lineWidth = 2),
    data = map(seq_len(nrow(fwd_year)), ~ list(
      fwd_year$fiscal_year[.x],
      round(fwd_year$mean_fwd[.x], 2)
    ))
  ) |>
  hc_chart(height = 380)
```

> **Interpretation.** Higher forward-looking share may correlate with management confidence; drops signal caution or elevated macro uncertainty. Wide IQR bands reflect heterogeneous guidance cultures across firms. Extend with a fine-tuned BERT classifier (trained on 10-K Item 7 forward-looking sentences) for higher precision than regex.

### 3.8 Sentiment — Loughran-McDonald Lexicon

The LM lexicon is purpose-built for financial text, avoiding false positives common in generic tools (e.g. *liability* ≠ negative in general English).

```{r sentiment}
# ── Net sentiment ribbon (hc_add_series required for arearange overlay) ────────
sent_time <- sentiment_doc |>
  group_by(fiscal_year) |>
  summarise(
    mean_net = mean(net, na.rm = TRUE),
    q25      = quantile(net, 0.25, na.rm = TRUE),
    q75      = quantile(net, 0.75, na.rm = TRUE),
    .groups  = "drop"
  )

hc_net <- highchart() |>
  hc_add_theme(my_theme) |>
  hc_chart(zoomType = "x") |>
  hc_title(text = "Net Sentiment Over Time (Loughran-McDonald)") |>
  hc_subtitle(text = "(positive − negative) / total sentiment words — mean ± IQR") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(
    title = list(text = "Net sentiment"),
    plotLines = list(list(value = 0, color = "#2a2a42", width = 1, dashStyle = "Dash"))
  ) |>
  hc_tooltip(shared = TRUE, crosshairs = TRUE, valueDecimals = 3) |>
  hc_add_series(
    name        = "IQR",
    type        = "arearange",
    color       = "#34d399",
    fillOpacity = 0.12,
    lineWidth   = 0,
    marker      = list(enabled = FALSE),
    data = map(seq_len(nrow(sent_time)), ~ list(
      sent_time$fiscal_year[.x],
      round(sent_time$q25[.x], 4),
      round(sent_time$q75[.x], 4)
    ))
  ) |>
  hc_add_series(
    name      = "Mean net sentiment",
    type      = "spline",
    color     = "#34d399",
    lineWidth = 2.5,
    marker    = list(radius = 4, symbol = "circle",
                     fillColor = "#34d399", lineColor = "#0d0d14", lineWidth = 2),
    data = map(seq_len(nrow(sent_time)), ~ list(
      sent_time$fiscal_year[.x],
      round(sent_time$mean_net[.x], 4)
    ))
  ) |>
  hc_chart(height = 340)

# ── Sentiment category breakdown ───────────────────────────────────────────────
cats <- c("positive", "negative", "uncertainty", "litigious", "constraining")

cat_time <- tokens |>
  inner_join(lm_sentiment, by = "word") |>
  filter(sentiment %in% cats) |>
  count(fiscal_year, sentiment) |>
  group_by(fiscal_year) |>
  mutate(pct = n / sum(n) * 100) |>
  ungroup() |>
  mutate(sentiment = factor(sentiment, levels = cats))

hc_cats <- hchart(cat_time, "column", hcaes(x = fiscal_year, y = pct, group = sentiment)) |>
  hc_colors(unname(cat_palette[cats])) |>
  hc_add_theme(my_theme) |>
  hc_plotOptions(column = list(
    stacking     = "normal",
    borderWidth  = 0,
    groupPadding = 0.05,
    pointPadding = 0.02
  )) |>
  hc_title(text = "Sentiment Category Breakdown") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title  = list(text = "% sentiment words"),
           labels = list(format = "{value}%"),
           max    = 100) |>
  hc_tooltip(shared = TRUE, valueSuffix = "%", valueDecimals = 1) |>
  hc_chart(height = 340)


```

:::{layout-ncol=2}
```{r}
hc_net
```
```{r}
hc_cats
```
:::



> **Interpretation.** Net sentiment dips tend to cluster around market downturns (2020, 2022). Rising *uncertainty* share is a leading signal of guidance risk. The LM *litigious* category often spikes around regulatory cycles. Next step: FinBERT sentence-level scoring for contextual sentiment that goes beyond lexical matching.

### 3.9 Market Reaction — Close-to-Open Return

```{r market-reaction}
# Distribution by year (mean ± IQR) — top-50 firms with price data
ret_year <- earnings_calls_raw |>
  filter(!is.na(close_to_open_return)) |>
  group_by(fiscal_year) |>
  summarise(
    mean_ret = mean(close_to_open_return) * 100,
    q25      = quantile(close_to_open_return, 0.25) * 100,
    q75      = quantile(close_to_open_return, 0.75) * 100,
    .groups  = "drop"
  )

hc_ret <- hchart(ret_year, "column", hcaes(x = fiscal_year, y = mean_ret),
                 name = "Mean Return (%)",
                 color = "#a78bfa",
                 borderRadius = 3,
                 dataLabels = list(
                   enabled = TRUE,
                   format  = "{y:.2f}%",
                   style   = list(fontSize = "9px", color = "#e8e8f0",
                                  textOutline = "none")
                 )) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Mean Close-to-Open Return on Earnings Day") |>
  hc_subtitle(text = "Top-50 S&P 500 firms with price data") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(
    title = list(text = "Return (%)"),
    labels = list(format = "{value}%"),
    plotLines = list(list(value = 0, color = "#2a2a42",
                          width = 1, dashStyle = "Dash"))
  ) |>
  hc_tooltip(pointFormat = "<b>{point.x}</b>: {point.y:.3f}%")

# Distribution scatter — shows dispersion, not just mean
ret_scatter <- earnings_calls |>
  filter(!is.na(close_to_open_return)) |>
  mutate(ret_pct = close_to_open_return * 100) |>
  select(fiscal_year, ticker, ret_pct)

hc_scatter <- hchart(ret_scatter, "scatter",
                     hcaes(x = fiscal_year, y = ret_pct, group = ticker),
                     marker = list(radius = 2, symbol = "circle"),
                     showInLegend = FALSE) |>
  hc_add_theme(my_theme) |>
  hc_title(text = "Return Distribution by Year") |>
  hc_subtitle(text = "Each point = one earnings call") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title  = list(text = "Close-to-Open Return (%)"),
           labels = list(format = "{value}%"),
           plotLines = list(list(value = 0, color = "#2a2a42",
                                 width = 1, dashStyle = "Dash"))) |>
  hc_tooltip(pointFormat = "<b>{point.ticker}</b><br>{point.y:.2f}%") |>
  hc_chart(height = 360)

# tagList(tags$div(
#   style = "display:grid; grid-template-columns:1fr 1fr; gap:12px;",
#   hc_ret, hc_scatter
# ))
```


:::{layout-ncol=2}
```{r}
hc_ret
```
```{r}
hc_scatter
```
:::


> **Interpretation.** The close-to-open return captures the overnight market reaction to the earnings call. Connecting this to sentiment, forward-looking language share, or topic dominance is a natural starting point for research. High variance years (visible in the scatter) may reflect macro volatility compressing post-earnings drift.

---

## 4 · Python NLP

### 4.1 Basic Validation & Frequency Analysis

In [ ]:
#| label: py-nltk
#| code-fold: false

import pandas as pd
import numpy as np

tok = r.tokens[['ticker', 'word', 'fiscal_year']].copy()

# ── Scalars ───────────────────────────────────────────────────────────────────
total_tokens = int(len(tok))
unique_terms = int(tok['word'].nunique())

# ── Top 25 terms — parallel lists ─────────────────────────────────────────────
top25         = tok['word'].value_counts().head(25)
top25_words   = top25.index.tolist()
top25_counts  = [int(x) for x in top25.values.tolist()]

# ── Tokens by firm (top 10) — parallel lists ──────────────────────────────────
by_firm       = tok.groupby('ticker').size().sort_values(ascending=False).head(10)
firm_tickers  = by_firm.index.tolist()
firm_counts   = [int(x) for x in by_firm.values.tolist()]

```{r py-nltk-render}
#| echo: false

tagList(
  tags$p(
    tags$strong(paste0("Total tokens (top-50): ", format(py$total_tokens, big.mark = ","))),
    tags$span("  ·  "),
    tags$strong(paste0("Unique terms: ", format(py$unique_terms, big.mark = ",")))
  ),
  tags$div(
    style = "display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-top:12px;",
    tags$div(
      tibble(Term = py$top25_words, Frequency = as.integer(py$top25_counts)) |>
        kable(caption = "Top 25 Terms by Frequency") |>
        kable_styling(bootstrap_options = c("striped","hover","condensed"),
                      full_width = TRUE, font_size = 12) |>
        htmltools::HTML()
    ),
    tags$div(
      tibble(Ticker = py$firm_tickers, `Token Count` = as.integer(py$firm_counts)) |>
        kable(caption = "Token Count by Firm (Top 10)") |>
        kable_styling(bootstrap_options = c("striped","hover","condensed"),
                      full_width = TRUE, font_size = 12) |>
        htmltools::HTML()
    )
  )
)
```


### 4.2 FinBERT Sentiment Scoring

> POC on first 1000 tokens in each of top 50 earnings calls

```{r finbert-prep}
#| echo: false
#| cache: false
#| cache.lazy: false

# Prepare plain vectors — no Date columns, no dataframe conversion
.fb_texts   <- earnings_calls$full_transcript_text
.fb_tickers <- earnings_calls$ticker
.fb_years   <- as.numeric(earnings_calls$fiscal_year)
```
```{r finbert-check}
#| cache: false
#| echo: false
here_path      <- here::here('analytical_ml_finance')
finbert_exists <- file.exists(file.path(here_path,"finbert_results.pkl"))

```

In [ ]:
#| label: py-finbert
#| eval: true
#| code-fold: false

import pickle
import os
import random
import pandas as pd
import numpy as np
from pathlib import Path

pkl_path = Path(r.here_path) / "finbert_results.pkl"

if r.finbert_exists:
    # ── Load cached results ────────────────────────────────────────────────────
    with open(pkl_path, 'rb') as f:
        res = pickle.load(f)
    print(f"Loaded cached FinBERT results: {len(res)} rows")

else:
    # ── Run inference — all top-50 transcripts, first 1000 chars ──────────────
    from transformers import pipeline

    texts_raw   = list(r[".fb_texts"])
    tickers_raw = list(r[".fb_tickers"])
    years_raw   = list(r[".fb_years"])

    combined = [
        (str(t), str(tk), float(y) if (y is not None and not (isinstance(y, float) and y != y)) else None)
        for t, tk, y in zip(texts_raw, tickers_raw, years_raw)
        if t is not None and isinstance(t, str) and len(t) > 100
    ]

    sample_texts   = [row[0][0:2000] for row in combined]
    sample_tickers = [row[1]        for row in combined]
    sample_years   = [row[2]        for row in combined]

    finbert = pipeline(
        "text-classification",
        model      = "ProsusAI/finbert",
        tokenizer  = "ProsusAI/finbert",
        top_k      = None,
        truncation = True,
        max_length = 512,
        device     = "mps"   # change to 0 for CUDA, -1 for CPU
    )

    results = finbert(sample_texts)

    rows = []
    for ticker, year, scores in zip(sample_tickers, sample_years, results):
        if isinstance(scores, dict):
            scores = [scores]
        best = max(scores, key=lambda x: x['score'])
        rows.append({
            'ticker'     : ticker,
            'fiscal_year': int(year) if year is not None else None,
            'label'      : best['label'],
            'score'      : round(best['score'], 3)
        })

    res = pd.DataFrame(rows)

    with open(pkl_path, 'wb') as f:
        pickle.dump(res, f)
        print(f"FinBERT inference complete. Saved {len(res)} rows to {pkl_path}")

# ── Aggregates → parallel lists for R ─────────────────────────────────────────
label_counts  = res['label'].value_counts()
fb_agg_labels = label_counts.index.tolist()
fb_agg_counts = [int(x) for x in label_counts.values.tolist()]

year_label   = (
    res.dropna(subset=['fiscal_year'])
       .groupby(['fiscal_year', 'label'])
       .size()
       .reset_index(name='n')
)
fb_yr_years  = [int(x) for x in year_label['fiscal_year'].tolist()]
fb_yr_labels = year_label['label'].tolist()
fb_yr_counts = [int(x) for x in year_label['n'].tolist()]

# Net sentiment by year (positive - negative) / total
net_sent = (
    res.dropna(subset=['fiscal_year'])
       .groupby(['fiscal_year', 'label'])
       .size()
       .unstack(fill_value=0)
       .reset_index()
)
for col in ['positive', 'negative', 'neutral']:
    if col not in net_sent.columns:
        net_sent[col] = 0

net_sent['net']    = (net_sent['positive'] - net_sent['negative']) / \
                     (net_sent['positive'] + net_sent['negative'] + net_sent['neutral'])
fb_net_years  = [int(x) for x in net_sent['fiscal_year'].tolist()]
fb_net_scores = [round(float(x), 4) for x in net_sent['net'].tolist()]

```{r py-finbert-render}
#| echo: false
#| eval: true

fb_pal <- c(positive = "#34d399", negative = "#f87171", neutral = "#f5a623")

# ── Overall label distribution ─────────────────────────────────────────────────
agg_df <- tibble(
  label = py$fb_agg_labels,
  n     = as.integer(py$fb_agg_counts)
) |> mutate(label = factor(label, levels = c("positive", "neutral", "negative")))

hc_fb_agg <- hchart(agg_df, "column", hcaes(x = label, y = n, color = label),
                    name = "Transcripts", borderRadius = 3,
                    dataLabels = list(
                      enabled = TRUE, format = "{y}",
                      style   = list(fontSize = "11px", color = "#e8e8f0",
                                     textOutline = "none")
                    )) |>
  hc_colors(unname(fb_pal[levels(agg_df$label)])) |>
  hc_add_theme(my_theme) |>
  hc_title(text    = "FinBERT Sentiment Distribution") |>
  hc_subtitle(text = paste0("n = ", sum(agg_df$n), " transcripts (top-50 firms)")) |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title = list(text = "# Transcripts")) |>
  hc_chart(height = 340) |>
  hc_tooltip(pointFormat = "<b>{point.x}</b>: {point.y} transcripts")

# ── Stacked sentiment by year ──────────────────────────────────────────────────
yr_df <- tibble(
  fiscal_year = as.integer(py$fb_yr_years),
  label       = py$fb_yr_labels,
  n           = as.integer(py$fb_yr_counts)
) |> mutate(label = factor(label, levels = c("positive", "neutral", "negative")))

hc_fb_yr <- hchart(yr_df, "column", hcaes(x = fiscal_year, y = n, group = label)) |>
  hc_colors(unname(fb_pal[c("positive", "neutral", "negative")])) |>
  hc_add_theme(my_theme) |>
  hc_plotOptions(column = list(stacking = "normal", borderWidth = 0)) |>
  hc_title(text    = "FinBERT Sentiment by Year") |>
  hc_subtitle(text = "Stacked count — positive / neutral / negative") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title = list(text = "# Transcripts")) |>
  hc_chart(height = 340) |>
  hc_tooltip(shared = TRUE)

# ── Net sentiment over time (mirrors LM chart) ─────────────────────────────────
net_df <- tibble(
  fiscal_year = as.integer(py$fb_net_years),
  net         = as.numeric(py$fb_net_scores)
)

hc_fb_net <- hchart(net_df, "spline", hcaes(x = fiscal_year, y = net),
                    name = "Net sentiment", color = "#34d399",
                    lineWidth = 2.5,
                    marker = list(radius = 4, symbol = "circle",
                                  fillColor = "#34d399",
                                  lineColor = "#0d0d14", lineWidth = 2)) |>
  hc_add_theme(my_theme) |>
  hc_title(text    = "FinBERT Net Sentiment Over Time") |>
  hc_subtitle(text = "(positive − negative) / total — top-50 firms, first 1000 chars") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(
    title     = list(text = "Net sentiment"),
    plotLines = list(list(value = 0, color = "#2a2a42",
                          width = 1, dashStyle = "Dash"))
  ) |>
  hc_tooltip(pointFormat = "<b>{point.x}</b>: {point.y:.4f}") |>
  hc_chart(height = 340)
```

:::{layout-ncol=2}
```{r}
#| eval: true
#| echo: false
hc_fb_agg
```
```{r}
#| eval: true
#| echo: false
hc_fb_yr
```
:::
```{r}
#| eval: true
#| echo: false
hc_fb_net
```

### 4.3 Gensim LDA

*Set `eval: true` after `pip install gensim`. Mirrors the R LDA for cross-validation of topic coherence.*

In [ ]:
#| label: py-gensim
#| eval: false
#| code-fold: false

# pip install gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pandas as pd

# Build per-document word lists from R tokens
tok_py = r.tokens[['doc_id', 'word']].copy()
docs   = tok_py.groupby('doc_id')['word'].apply(list).tolist()

# Build dictionary and bag-of-words corpus
dictionary   = corpora.Dictionary(docs)
dictionary.filter_extremes(no_below=10, no_above=0.85)
bow_corpus   = [dictionary.doc2bow(doc) for doc in docs]

print(f"Vocabulary size after pruning: {len(dictionary):,}")
print(f"Number of documents          : {len(bow_corpus):,}")

# Train LDA (k=8 to match R model)
lda = LdaModel(
    corpus       = bow_corpus,
    id2word      = dictionary,
    num_topics   = 8,
    random_state = 42,
    passes       = 10,
    alpha        = 'auto',
    eta          = 'auto'
)

print("\nTop-10 terms per topic:")
for i, topic in lda.print_topics(num_topics=8, num_words=10):
    print(f"  Topic {i+1}: {topic}")

# Coherence score (C_v) — higher is better; tune k over 4:14
coherence = CoherenceModel(
    model      = lda,
    texts      = docs,
    dictionary = dictionary,
    coherence  = 'c_v'
).get_coherence()

print(f"\nC_v coherence score (k=8): {coherence:.4f}")

---

## Appendix · Sentence Embeddings

Sentence-level embeddings via `all-MiniLM-L6-v2` → UMAP(2D) → K-means. **Set `eval: true` in the Python chunk below** after configuring the Python environment.

```{r embed-placeholder}
#| echo: false

# R placeholder rendered until Python env is configured
highchart() |>
  hc_add_theme(my_theme) |>
  hc_chart(type = "scatter") |>
  hc_title(text = "UMAP — Transcript Embedding Space") |>
  hc_subtitle(text = "Set eval: true in py-embed chunk after: pip install sentence-transformers umap-learn") |>
  hc_xAxis(title = list(text = "UMAP-1"), min = -5, max = 5) |>
  hc_yAxis(title = list(text = "UMAP-2"), min = -5, max = 5) |>
  hc_add_series(
    name  = "Pending",
    color = "#2a2a42",
    data  = list(list(0, 0)),
    marker = list(radius = 3)
  ) |>
  hc_annotations(list(
    labels = list(list(
      point = list(x = 0, y = 0, xAxis = 0, yAxis = 0),
      text  = "Awaiting embeddings",
      style = list(color = "#e8e8f0", fontSize = "13px")
    ))
  ))
```

In [ ]:
#| label: py-embed
#| eval: false
#| code-fold: false

# pip install sentence-transformers umap-learn
from sentence_transformers import SentenceTransformer
import umap
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# Sample transcripts — first 1,200 chars to fit model context
sample = (
    r.earnings_calls_raw[['ticker', 'fiscal_year', 'full_transcript_text']]
     .sample(min(500, len(r.earnings_calls_raw)), random_state=42)
     .copy()
)
sample['chunk'] = sample['full_transcript_text'].str[:1200]

embedder   = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(
    sample['chunk'].tolist(),
    batch_size          = 64,
    normalize_embeddings = True,
    show_progress_bar    = True
)

reducer = umap.UMAP(n_components=2, random_state=42, metric='cosine')
coords  = reducer.fit_transform(embeddings)

sample['umap1']   = coords[:, 0]
sample['umap2']   = coords[:, 1]
sample['cluster'] = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(coords)

print("Cluster sizes:")
print(sample['cluster'].value_counts().sort_index().to_string())
print("\nSample points:")
print(sample[['ticker','fiscal_year','cluster','umap1','umap2']].head(10).to_string(index=False))

# Hand off to R for highcharter rendering:
# embed_df <- py$sample  — then use hchart(embed_df, "scatter", hcaes(umap1, umap2, group=cluster))

**To render the UMAP in highcharter after running the Python chunk:**

```r
# embed_df <- py$sample |> mutate(cluster = factor(cluster))
# hchart(embed_df, "scatter", hcaes(x = umap1, y = umap2, group = cluster, name = ticker)) |>
#   hc_add_theme(my_theme) |>
#   hc_title(text = "UMAP — Transcript Embedding Space") |>
#   hc_subtitle(text = "all-MiniLM-L6-v2 → UMAP(2D) → K-means (k=6)") |>
#   hc_xAxis(title = list(text = "UMAP-1")) |>
#   hc_yAxis(title = list(text = "UMAP-2")) |>
#   hc_tooltip(pointFormat = "<b>{point.name}</b><br>Cluster {series.name}<br>({point.x:.2f}, {point.y:.2f})")
```